In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools

import sklearn.metrics
import sklearn.decomposition

np.set_printoptions(suppress=True)

In [2]:
import glob
from scipy.interpolate import interp1d

### 1. Loading pretrained model

In [4]:
vae.summary()

Model: "vae"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 2780, 1)]    0           []                               
                                                                                                  
 xrd_reshape (Reshape)          (None, 139, 20)      0           ['input_1[0][0]']                
                                                                                                  
 LC_xrd (Dense)                 (None, 139, 160)     3360        ['xrd_reshape[0][0]']            
                                                                                                  
 LN_xrdReshape (LayerNormalizat  (None, 139, 160)    320         ['LC_xrd[0][0]']                 
 ion)                                                                                           

In [5]:
n_hidden = 139*5

vae = tf.keras.models.load_model('./vae.h5')
vae = tf.keras.Model(inputs=vae.inputs, outputs = vae.outputs, name='vae')
vae.trainable=False

encoder = tf.keras.Model(inputs=vae.inputs, outputs = vae.get_layer('mean_layer').output, name='encoder')
decoder = tf.keras.Model(inputs = vae.get_layer('LC1_decoder').input, outputs = vae.get_layer('activation').output, name='decoder')

### 2. Loading dataset

In [6]:
records = glob.glob('../experimental_xrd/*.csv')
name_samples = {k:v.split('\\')[1].split('-interpolated.')[0] for k,v in enumerate(records)}

x = list()
for record in records:
    
    x += [pd.read_csv(record).iloc[:,1].values]

x = np.stack(x)
x = x[:,:-1]
x = x/np.max(x, axis=1, keepdims=True)
x = np.expand_dims(x, axis=-1)

In [7]:
dtheta = pd.read_csv(records[-1]).iloc[:-1,0].values

### 3. Encoding-reconstruction

In [8]:
ls = encoder(x, training=False)
xrec = decoder(ls, training = False)

### 4. Different compounds

In [11]:
def interpolate2(om=0, om2 = 1, steps = 5, show=False):

    interpolated = list()
    c_phase1 = list()
    c_phase2 = list()
    for alpha in np.linspace(0,1,steps):
        phase1 = alpha*ls[om, :]
        phase2 = (1-alpha)*ls[om2, :]
        
        interpolated += [phase1 + phase2]
        c_phase1 += [phase1]
        c_phase2 += [phase2]
    interpolated = np.stack(interpolated)
    c_phase1 = np.stack(c_phase1)
    c_phase2 = np.stack(c_phase2)
        
    return np.stack((interpolated, c_phase1, c_phase2))

In [12]:
for tupla in list(itertools.combinations(range(6),2)):
    mixed_phases = interpolate2(tupla[0], tupla[1], steps=51)
    np.save(f"mixed_phases/ls-{name_samples[tupla[0]]}_{name_samples[tupla[1]]}", mixed_phases)

In [11]:
stepsize = 2

samplings = list()

interpolated = list()
c_phase1 = list()
c_phase2 = list()
c_phase3 = list()
for n in np.arange(0,100 + stepsize, stepsize):
    sm = 100-n
    for s in np.arange(0,sm + stepsize, stepsize):
        samplings += [(n, s, sm-s)]
        
        phase1 = (n*1e-2)*ls[4, :]
        phase2 = (s*1e-2)*ls[0, :]
        phase3 = (sm-s)*(1e-2)*ls[5, :]
        
        interpolated += [phase1 + phase2 + phase3]
        c_phase1 += [phase1]
        c_phase2 += [phase2]
        c_phase3 += [phase3]
        
interpolated = np.stack(interpolated)
c_phase1 = np.stack(c_phase1)
c_phase2 = np.stack(c_phase2)
c_phase3 = np.stack(c_phase3)

xrec = decoder(interpolated, training=False)
x_phase1 = decoder(c_phase1, training=False)
x_phase2 = decoder(c_phase2, training=False)
x_phase3 = decoder(c_phase3, training=False)

triple_phase = np.stack((xrec.numpy(), x_phase1.numpy(), x_phase2.numpy(), x_phase3.numpy()))
np.save('triple_phase', triple_phase)
pd.DataFrame(samplings).to_csv('sampling_triple_phase.csv', index=None)